# Merge: verify the five notebooks and compute verdicts

Runs on CPU in minutes. This is the only place sealed test metrics are computed. A sealed dataset stays sealed until its planned sweep jobs have all finished, and every unsealing is logged. **Do not rerun on a changed configuration and report the new numbers as confirmation:** a rerun under different code is labelled exploratory.

**On Kaggle, attach as inputs:** the code package, each notebook's output dataset (`aml-out-nbX-vN`), and the raw datasets.

**On Colab:** put the code package in Drive `aml`, and copy every notebook's output folder (`out_nb1` and so on) into Drive `aml/collect`. The raw files are downloaded for you. A free runtime has about 12 GB of RAM, so by default only HI-Small and LI-Small are merged there. Add the Medium datasets to `DATASETS` only on a larger runtime.

In [ ]:
import os, sys
NOTEBOOK = 'merge'
SHARD = None
_COLAB = bool(os.environ.get('AMLH_PLATFORM') == 'colab' or 'google.colab' in sys.modules)
DATASETS = ['HI-Small', 'LI-Small'] if _COLAB else ['HI-Small', 'LI-Small', 'SAML-D', 'HI-Medium', 'LI-Medium']
EXPECTED_CODE_HASH = '39a6ee5169e533f6'
INSTALL = True

In [ ]:
# Where things live. Kaggle attaches datasets under /kaggle/input. Colab has none, so it downloads the raw files and keeps its output in Google Drive,
# which means a disconnected session resumes from where it stopped.
import os, sys
from pathlib import Path

PLATFORM = os.environ.get('AMLH_PLATFORM') or ('colab' if 'google.colab' in sys.modules else 'kaggle')
CONTENT = Path('/content')
DRIVE = CONTENT / 'drive' / 'MyDrive'
if PLATFORM == 'colab':
    if not DRIVE.is_dir():
        from google.colab import drive
        drive.mount(str(CONTENT / 'drive'))
    HOME = DRIVE / 'aml'                       # put the code package (aml-code-*.zip) here
    INPUT = CONTENT / 'aml_data'               # raw files, downloaded at the start of each session
    CODE_SEARCH = [HOME, CONTENT]
    OUTPUTS = HOME / 'collect'                 # the merge notebook reads every notebook's output from here
    WORKING = HOME / (f"out_{NOTEBOOK.lower()}" + (f"_shard{SHARD[0]}of{SHARD[1]}" if SHARD else ''))
    RESTORE = []                               # the output already stays in Drive
    WORKING.mkdir(parents=True, exist_ok=True)
else:
    INPUT = Path('/kaggle/input')
    CODE_SEARCH = [INPUT]
    OUTPUTS = INPUT
    WORKING = Path('/kaggle/working')
    RESTORE = None                             # attached earlier outputs are found under /kaggle/input
print(PLATFORM, 'input', INPUT, 'output', WORKING)

In [ ]:
# Unpacks the code package outside the output folder, so it is not exported. Kaggle may unzip an uploaded zip, so both forms are handled.
import glob, shutil, subprocess, sys, time, zipfile

CODE = Path('/tmp/aml_code')
shutil.rmtree(CODE, ignore_errors=True)
if PLATFORM == 'colab':
    found = {p.name: p for root in CODE_SEARCH for p in sorted(Path(root).glob('aml-code-*.zip'))}  # the same file in two places counts once
    zips, folders = [str(p) for p in found.values()], []
else:
    zips = glob.glob(str(INPUT / '**' / 'aml-code-*.zip'), recursive=True)
    folders = [p.parents[2] for p in INPUT.glob('**/src/amlh/runner.py')]
assert len(zips) + len(folders) == 1, f'attach or upload exactly one code package; found {zips + [str(f) for f in folders]}'
if zips:
    zipfile.ZipFile(zips[0]).extractall(CODE)
else:
    shutil.copytree(folders[0], CODE)
sys.path.insert(0, str(CODE / 'src'))

from amlh.runner import code_hash
assert code_hash(CODE) == EXPECTED_CODE_HASH, f'this notebook expects code {EXPECTED_CODE_HASH}, but the package is {code_hash(CODE)}'
if INSTALL:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements-kaggle.txt')], check=True)
print('code version', EXPECTED_CODE_HASH)

In [ ]:
if PLATFORM == 'colab':
    from amlh.fetch import ensure_raw
    ensure_raw(DATASETS, INPUT, backup=HOME / 'data')

In [ ]:
from amlh.jobs_gbt import Paths
from amlh.merge import merge
from amlh.session import find_raw, verify_raw

available = [d for d in DATASETS if list(INPUT.rglob(f'{d}_Trans.csv')) or (d == 'SAML-D' and list(INPUT.rglob('SAML-D.csv')))]
Paths.raw = find_raw(available, INPUT)
Paths.cache = Path('/tmp/aml_cache')
Paths.cache.mkdir(parents=True, exist_ok=True)
for d in available:
    print(verify_raw(d, Paths.raw[d]))

In [ ]:
OUT = WORKING / 'merged'
report = merge([OUTPUTS], OUT, available, analysis_root=CODE)
print((OUT / 'MERGE_REPORT.md').read_text())

## Tables and figures

`08_analyze.py` regenerates every table, figure and in-text number from the exported results. It currently covers the Small datasets. Its output is copied to `merged/report`.

In [ ]:
os.makedirs(CODE / 'data', exist_ok=True)
link = CODE / 'data' / 'processed'
if not link.exists():
    os.symlink(Paths.cache, link)
subprocess.run([sys.executable, str(CODE / 'scripts' / '02b_audit_all.py')], check=False)
result = subprocess.run([sys.executable, str(CODE / 'scripts' / '08_analyze.py'), str(OUT)], capture_output=True, text=True)
print(result.stdout[-1500:], result.stderr[-1500:])
if (CODE / 'results' / 'report').exists():
    shutil.copytree(CODE / 'results' / 'report', OUT / 'report', dirs_exist_ok=True)